# Pandas — Full Reference

**Context:** Phase 1, Day 2 — Placement prep for AI Engineer roles.

Pandas is the standard tool for loading, inspecting, and preprocessing tabular data before it ever touches a model.
Everything you learned in NumPy transfers — pandas is built on top of it.

**Dataset used:** `students.csv` — a small placement-style dataset (features: cgpa, internships, skills → target: placed)

---
## 1. Importing Pandas

`pd` is the universal alias. Always import NumPy alongside it — you'll use both together constantly in ML pipelines.

In [ ]:
import pandas as pd
import numpy as np

print("pandas:", pd.__version__)
print("numpy: ", np.__version__)

---
## 2. Core Data Structures: Series & DataFrame

Pandas has two structures:
- **Series** — 1D labeled array (like a single column with an index). Think: `np.array` + named index.
- **DataFrame** — 2D table with labeled rows AND columns. Think: the main structure for every dataset you'll ever load.

### Series
A Series is a NumPy array with an index. The index makes lookups by label possible — not just by position.

In [ ]:
cgpa = pd.Series([8.5, 7.2, 9.1, 6.8], index=['Alice', 'Bob', 'Charlie', 'Diana'])
print(cgpa)
print()
print("Alice's CGPA:", cgpa['Alice'])
print("\nCGPA > 8.0:\n", cgpa[cgpa > 8.0])

In [ ]:
print("dtype:  ", cgpa.dtype)
print("shape:  ", cgpa.shape)
print("values: ", cgpa.values)
print("index:  ", cgpa.index.tolist())

### DataFrame
A DataFrame is a collection of Series sharing the same index. Every column is a Series. This is the structure you'll work with 95% of the time.

In [ ]:
df = pd.DataFrame({
    'name':        ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve'],
    'cgpa':        [8.5, 7.2, 9.1, 6.8, 8.9],
    'internships': [2, 1, 3, 0, 2],
    'placed':      [1, 0, 1, 0, 1]
})
df

In [ ]:
print("shape:  ", df.shape)
print("columns:", df.columns.tolist())
print("dtypes:\n", df.dtypes)
print("\nsize:   ", df.size)

---
## 3. Loading & Inspecting Data

In real ML work you almost always start from a file, not a hand-crafted dict. `pd.read_csv` is the most commonly used function in data science — period.

In [ ]:
df = pd.read_csv('../students.csv')
df

In [ ]:
df.head()

In [ ]:
df.tail(3)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print("Placement distribution:\n", df['placed'].value_counts())
print()
print("Primary skills:\n", df['primary_skill'].value_counts())

---
## 4. Selecting & Indexing

Pandas has three selection methods:

| Method | What it does | Use when |
|--------|-------------|----------|
| `df['col']` | Select column(s) by name | Selecting features |
| `df.loc[rows, cols]` | Select by **label** | Named access |
| `df.iloc[rows, cols]` | Select by **integer position** | Positional slicing |

### Column Selection
Single brackets → Series. Double brackets → DataFrame.

In [ ]:
cgpa_col = df['cgpa']
print(type(cgpa_col))
print(cgpa_col.head())

In [ ]:
features = df[['cgpa', 'internships', 'projects']]
print(type(features))
features.head()

In [ ]:
# In ML — split features (X) from target (y)
X = df[['cgpa', 'internships', 'projects']]
y = df['placed']

print("X shape:", X.shape)
print("y shape:", y.shape)

### `.loc` — Label-based Selection
Both ends of the slice are **inclusive**.

In [ ]:
print(df.loc[0])
print()
print(df.loc[0, 'cgpa'])
print()
print(df.loc[0:3, 'cgpa':'internships'])

In [ ]:
placed_students = df.loc[df['placed'] == 1, ['name', 'cgpa', 'internships']]
placed_students

### `.iloc` — Integer-position Selection
End index is **exclusive** — exactly like NumPy.

In [ ]:
print(df.iloc[0])
print()
print(df.iloc[0, 2])
print()
print(df.iloc[0:3, 1:4])

In [ ]:
# Key difference — loc includes end, iloc excludes end
print("loc  0:2:", df.loc[0:2, 'name'].tolist())   # rows 0, 1, 2
print("iloc 0:2:", df.iloc[0:2]['name'].tolist())  # rows 0, 1 only

### Boolean Filtering
Same concept as NumPy boolean masking. Wrap each condition in parentheses.

In [ ]:
strong_candidates = df[(df['cgpa'] >= 8.5) & (df['internships'] >= 2)]
strong_candidates[['name', 'cgpa', 'internships', 'placed']]

In [ ]:
python_devs = df[df['primary_skill'].isin(['Python'])]
print(f"Python devs: {len(python_devs)}")

non_python = df[~df['primary_skill'].isin(['Python'])]
print(f"Non-Python: {len(non_python)}")

---
## 5. Handling Missing Data

Real datasets are almost never clean. Missing values (`NaN`) must be handled before any ML model can use the data.

In [ ]:
df_messy = df.copy()
df_messy.loc[[2, 5, 11], 'cgpa'] = np.nan
df_messy.loc[[7, 15], 'internships'] = np.nan
df_messy.loc[[0], 'primary_skill'] = np.nan
df_messy.head(10)

In [ ]:
print("NaN counts per column:")
print(df_messy.isna().sum())
print(f"\nTotal NaNs: {df_messy.isna().sum().sum()}")
print(f"% missing: {df_messy.isna().mean().mul(100).round(1).to_dict()}")

In [ ]:
df_dropped = df_messy.dropna()
print(f"Original rows: {len(df_messy)}, After dropna: {len(df_dropped)}")

df_dropped_col = df_messy.dropna(subset=['cgpa'])
print(f"After dropna on cgpa only: {len(df_dropped_col)}")

In [ ]:
df_filled = df_messy.copy()

# Numeric — fill with mean
df_filled['cgpa'] = df_filled['cgpa'].fillna(df_filled['cgpa'].mean())
# Numeric — fill with median (robust to outliers)
df_filled['internships'] = df_filled['internships'].fillna(df_filled['internships'].median())
# Categorical — fill with mode
df_filled['primary_skill'] = df_filled['primary_skill'].fillna(df_filled['primary_skill'].mode()[0])

print("NaNs after filling:", df_filled.isna().sum().sum())

---
## 6. Data Manipulation

Transforming columns, creating new features, renaming, sorting — the core of feature engineering before model training.

### Creating New Columns (Feature Engineering)

In [ ]:
df_work = df.copy()

df_work['experience_score'] = df_work['internships'] * 10 + df_work['projects'] * 5
df_work['high_cgpa'] = (df_work['cgpa'] >= 8.5).astype(int)

df_work[['name', 'cgpa', 'internships', 'projects', 'experience_score', 'high_cgpa']].head()

### `.apply()` — Apply a function element-wise

In [ ]:
df_work['cgpa_grade'] = df_work['cgpa'].apply(lambda x: 'A' if x >= 8.5 else ('B' if x >= 7.0 else 'C'))
df_work['skill_type'] = df_work['primary_skill'].map({'Python': 'AI-ready', 'Java': 'Backend', 'C++': 'Systems'})

df_work[['name', 'cgpa', 'cgpa_grade', 'primary_skill', 'skill_type']].head(8)

### Sorting, Renaming, Dropping

In [ ]:
top_students = df.sort_values('cgpa', ascending=False).head(5)
top_students[['name', 'cgpa', 'internships', 'placed']]

In [ ]:
df_renamed = df.rename(columns={'cgpa': 'gpa', 'placed': 'hired'})
print(df_renamed.columns.tolist())

df_trimmed = df.drop(columns=['name'])
df_trimmed = df_trimmed.drop(index=[0, 1, 2])
print(df_trimmed.shape)

In [ ]:
df_work['placed'] = df_work['placed'].astype('float32')
print(df_work['placed'].dtype)

---
## 7. GroupBy & Aggregation

`groupby` splits the DataFrame into groups, applies a function, then combines results. Essential for EDA before building any model.

In [ ]:
df.groupby('placed')[['cgpa', 'internships', 'projects']].mean().round(2)

In [ ]:
df.groupby('primary_skill').agg(
    avg_cgpa=('cgpa', 'mean'),
    total_students=('name', 'count'),
    placed_count=('placed', 'sum'),
    avg_internships=('internships', 'mean')
).round(2)

In [ ]:
placement_rate = df.groupby('primary_skill')['placed'].mean().mul(100).round(1)
print("Placement rate by skill (%):\n", placement_rate)

---
## 8. Merging & Combining DataFrames

Combining data from multiple sources — joining feature tables, appending new batches, combining train/test splits.

In [ ]:
df_part1 = df.iloc[:10]
df_part2 = df.iloc[10:]

df_combined = pd.concat([df_part1, df_part2], ignore_index=True)
print(f"Part1: {df_part1.shape}, Part2: {df_part2.shape}, Combined: {df_combined.shape}")

In [ ]:
extra_info = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve', 'Frank', 'Grace', 'Henry', 'Iris', 'Jack'],
    'city': ['Mumbai', 'Delhi', 'Bangalore', 'Chennai', 'Hyderabad',
             'Pune', 'Bangalore', 'Mumbai', 'Delhi', 'Chennai'],
    'has_github': [1, 0, 1, 0, 1, 1, 1, 0, 1, 1]
})

df_merged = pd.merge(df, extra_info, on='name', how='inner')
df_merged[['name', 'cgpa', 'placed', 'city', 'has_github']].head()

In [ ]:
# Merge types:
# 'inner'  — only rows with matching keys in BOTH frames
# 'left'   — all rows from left, NaN where no match in right
# 'right'  — all rows from right, NaN where no match in left
# 'outer'  — all rows from both, NaN where no match

df_left = pd.merge(df, extra_info, on='name', how='left')
print(f"Left join: {df_left.shape}")
print(f"NaN cities: {df_left['city'].isna().sum()}")

---
## 9. Exporting Data

Save your processed DataFrame back to disk. Standard practice at the end of a preprocessing pipeline.

In [ ]:
# index=False prevents writing the row numbers as a column
df.to_csv('students_cleaned.csv', index=False)
print("Saved.")

# Other formats:
# df.to_json('data.json', orient='records')
# df.to_parquet('data.parquet')   ← columnar binary, standard in production

---
## 10. String Methods

Vectorized string operations via `.str` — apply string logic to an entire column without a loop.

In [ ]:
print(df['primary_skill'].str.upper().head())
print(df['name'].str.lower().head())

In [ ]:
# na=False prevents NaN values from raising an error
has_a = df[df['name'].str.contains('a', case=False, na=False)]
print("Names containing 'a':", has_a['name'].tolist())

In [ ]:
starts_A = df[df['name'].str.startswith('A')]
print("Names starting with A:", starts_A['name'].tolist())
print("\nName lengths:")
print(df['name'].str.len().head())

In [ ]:
df_str = df.copy()
df_str['skill_clean'] = df_str['primary_skill'].str.replace('Python', 'PY', regex=False)
print(df_str[['primary_skill', 'skill_clean']].head())

---
## 11. `query()` — Cleaner Boolean Filtering

Filter rows using a string expression — more readable for complex conditions.

In [ ]:
result_standard = df[(df['cgpa'] >= 8.5) & (df['internships'] >= 2)]
result_query    = df.query('cgpa >= 8.5 and internships >= 2')

print("Standard:", result_standard['name'].tolist())
print("query():  ", result_query['name'].tolist())
print("Same result:", result_standard.shape == result_query.shape)

In [ ]:
# Reference a Python variable inside query() with @
threshold = 8.0
skill = 'Python'

result = df.query('cgpa > @threshold and primary_skill == @skill')
result[['name', 'cgpa', 'primary_skill', 'placed']]

---
## 12. `rolling()`, `shift()`, `cumsum()` — Time-Series Operations

For ordered sequential data (stock prices, sensor readings, logs). Lower priority for Phase 1 — but useful to know.

In [ ]:
ts = pd.DataFrame({
    'day':   range(1, 11),
    'sales': [10, 14, 9, 18, 22, 15, 11, 25, 30, 20]
})
ts

In [ ]:
# Sliding window stats — first (n-1) rows are NaN (not enough data yet)
ts['rolling_3day_avg'] = ts['sales'].rolling(3).mean()
ts['rolling_3day_sum'] = ts['sales'].rolling(3).sum()
ts

In [ ]:
ts['prev_day_sales'] = ts['sales'].shift(1)      # lag feature — yesterday's value
ts['total_so_far']   = ts['sales'].cumsum()       # running total
ts['sales_rank']     = ts['sales'].rank(ascending=False).astype(int)

ts[['day', 'sales', 'prev_day_sales', 'total_so_far', 'sales_rank']]

---
## 13. Pandas ↔ NumPy Bridge

ML models (scikit-learn, PyTorch, TensorFlow) don't accept DataFrames — they want NumPy arrays.

In [ ]:
# The full preprocessing pipeline
X = df[['cgpa', 'internships', 'projects']]
y = df['placed']

X_array = X.to_numpy()
y_array = y.to_numpy()

print("X dtype:", X_array.dtype, "| shape:", X_array.shape)
print("y dtype:", y_array.dtype, "| shape:", y_array.shape)

X_norm = (X_array - X_array.min(axis=0)) / (X_array.max(axis=0) - X_array.min(axis=0))
print("\nNormalized X (first 3 rows):\n", X_norm[:3].round(3))

In [ ]:
# Wrap NumPy results back into a DataFrame for readability
X_norm_df = pd.DataFrame(X_norm, columns=['cgpa_norm', 'internships_norm', 'projects_norm'])
X_norm_df.head()

---
## Quick Reference Cheat Sheet

### Creating DataFrames
| Function | Description |
|---|---|
| `pd.DataFrame({...})` | From dictionary |
| `pd.read_csv('file.csv')` | From CSV file |
| `pd.Series([...])` | Create a 1D labeled array |

### Inspecting
| Function | Description |
|---|---|
| `df.head(n)` | First n rows |
| `df.tail(n)` | Last n rows |
| `df.info()` | Column types, null counts, memory |
| `df.describe()` | Stats: mean, std, min, max, quartiles |
| `df.shape` | (rows, cols) |
| `df.dtypes` | Data type of each column |
| `df['col'].value_counts()` | Frequency of each unique value |

### Selecting
| Syntax | Description |
|---|---|
| `df['col']` | Single column → Series |
| `df[['c1','c2']]` | Multiple columns → DataFrame |
| `df.loc[r, c]` | By label — end **inclusive** |
| `df.iloc[r, c]` | By position — end **exclusive** |
| `df[df['col'] > val]` | Boolean filter |
| `df[(c1) & (c2)]` | Compound condition |
| `df['col'].isin([...])` | Filter by list |

### Missing Data
| Function | Description |
|---|---|
| `df.isna().sum()` | Count NaNs per column |
| `df.dropna()` | Drop rows with any NaN |
| `df.dropna(subset=['col'])` | Drop only if specific col is NaN |
| `df['col'].fillna(val)` | Fill NaN with value |
| `df['col'].fillna(df['col'].mean())` | Fill with column mean |

### Transforming
| Function | Description |
|---|---|
| `df['new'] = df['a'] + df['b']` | Create new column |
| `df['col'].apply(func)` | Apply function element-wise |
| `df['col'].map({k: v})` | Map values to new values |
| `df.rename(columns={...})` | Rename columns |
| `df.drop(columns=[...])` | Drop columns |
| `df.sort_values('col')` | Sort by column |
| `df['col'].astype('float32')` | Change dtype |

### GroupBy
| Function | Description |
|---|---|
| `df.groupby('col').mean()` | Mean per group |
| `df.groupby('col').agg({...})` | Multiple aggregations |

### Combining
| Function | Description |
|---|---|
| `pd.concat([df1, df2])` | Stack vertically |
| `pd.merge(df1, df2, on='key', how='inner')` | SQL-style join |

### String Methods
| Function | Description |
|---|---|
| `df['col'].str.upper()` | Case conversion |
| `df['col'].str.contains('pat', na=False)` | Filter by pattern |
| `df['col'].str.startswith('x')` | Filter by prefix |
| `df['col'].str.replace('a', 'b')` | Replace substring |
| `df['col'].str.len()` | String length |

### query()
| Syntax | Description |
|---|---|
| `df.query('col > 5 and col2 == "x"')` | String expression filter |
| `df.query('col > @var')` | Use Python variable with `@` |

### Time-Series
| Function | Description |
|---|---|
| `df['col'].rolling(n).mean()` | Sliding window average |
| `df['col'].shift(n)` | Lag by n rows |
| `df['col'].cumsum()` | Running total |
| `df['col'].rank()` | Rank each value |

### NumPy Bridge
| Function | Description |
|---|---|
| `df.to_numpy()` | DataFrame → NumPy array |
| `pd.DataFrame(arr, columns=[...])` | NumPy → DataFrame |

### Export
| Function | Description |
|---|---|
| `df.to_csv('f.csv', index=False)` | Save to CSV |
| `df.to_parquet('f.parquet')` | Save to Parquet |

In [ ]:
# scratch